In [ ]:
import numpy as np
import pandas as pd
import scipy
import torch
import scanpy as sc
import cell2location
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import pyro.optim
import scipy.stats
import scipy.special
import os 
def create_hierarchical_annotations(sc_data):
    """Create hierarchical mapping between fine and coarse cell types"""
    print("Creating hierarchical cell type annotations...")
    
    # Dictionary to map fine cell types to coarse categories
    hierarchy_map = {}
    
    # Extract prefixes before first underscore to create coarse categories
    for cell_type in sc_data.obs['celltypes'].unique():
        if '_' in cell_type:
            prefix = cell_type.split('_')[0]
            hierarchy_map[cell_type] = prefix
        else:
            # If no underscore, keep as is (already coarse)
            hierarchy_map[cell_type] = cell_type
    
    # Add specific exceptions
    manual_adjustments = {
        'Hematopoeitic_stem_cell': 'Hematopoietic'
        # Add other exceptions if needed
    }
    hierarchy_map.update(manual_adjustments)
    
    # Apply hierarchical annotations
    sc_data.obs['coarse_celltype'] = sc_data.obs['celltypes'].map(hierarchy_map)
    
    # Create mapping from coarse to fine cell types
    coarse_to_fine = {}
    for fine, coarse in hierarchy_map.items():
        if coarse not in coarse_to_fine:
            coarse_to_fine[coarse] = []
        coarse_to_fine[coarse].append(fine)
    
    # Print summary
    print("\nHierarchical structure:")
    for coarse, fine_list in coarse_to_fine.items():
        print(f"{coarse}: {len(fine_list)} subtypes")
    
    return sc_data, hierarchy_map, coarse_to_fine

def preprocess_data(adata):
    """Basic preprocessing for Visium or single-cell data"""
    # Set counts as main data matrix if available
    if 'counts' in adata.layers:
        adata.X = adata.layers['counts'].copy()
    
    # Ensure counts are integers
    if scipy.sparse.issparse(adata.X):
        adata.X = adata.X.astype(np.int32)
    else:
        adata.X = np.round(adata.X).astype(np.int32)
    
    return adata

# Add before running deconvolution
def filter_problematic_genes(adata):
    """Filter out genes that might cause numerical instability"""
    # Filter genes with too many zeros
    sc.pp.filter_genes(adata, min_cells=10)
    
    # Filter genes with extreme expression values
    if scipy.sparse.issparse(adata.X):
        means = adata.X.mean(axis=0).A1
    else:
        means = adata.X.mean(axis=0)
    
    # Remove extreme high expression genes (might cause instability)
    high_expr_genes = adata.var_names[means > np.percentile(means, 99.9)]
    print(f"Removing {len(high_expr_genes)} extremely high expression genes")
    adata = adata[:, ~adata.var_names.isin(high_expr_genes)].copy()
    
    return adata

def load_and_split_data(visium_path, sc_path):
    """Load and split data by condition"""
    print("Loading and preprocessing data...")
    
    # Load Visium data
    visium = sc.read_h5ad(visium_path)
    visium = preprocess_data(visium)
    
    # Load single-cell data
    sc_data = sc.read_h5ad(sc_path)
    sc_data = preprocess_data(sc_data)
    
    # Add hierarchical annotations
    sc_data, hierarchy_map, coarse_to_fine = create_hierarchical_annotations(sc_data)
    
    # Split by condition
    visium_uninfected = visium[visium.obs['condition'] == 'uninfected'].copy()
    visium_infected = visium[visium.obs['condition'] == 'infected'].copy()
    
    sc_uninfected = sc_data[sc_data.obs['timepoint'] == '0wk'].copy()
    sc_infected = sc_data[sc_data.obs['timepoint'] == '3wk'].copy()
    
    # Rebuild categories for single-cell data
    for subset in [sc_uninfected, sc_infected]:
        subset_batches = subset.obs['batch'].astype(str).values
        subset.obs['batch'] = pd.Categorical(
            subset_batches,
            categories=np.unique(subset_batches),
            ordered=False
        )
        
        # Transfer the coarse_celltype annotation
        subset.obs['coarse_celltype'] = subset.obs['coarse_celltype'].astype('category')
    
    return visium_uninfected, visium_infected, sc_uninfected, sc_infected, hierarchy_map, coarse_to_fine

class AdaptiveConstraintModel(cell2location.models.Cell2location):
    def __init__(self, *args, **kwargs):
        # Extract custom parameters first
        self.constraint_scale = kwargs.pop('constraint_scale', 2.0)
        self.sparsity_scale = kwargs.pop('sparsity_scale', 0.2)
        self.orthogonality_scale = kwargs.pop('orthogonality_scale', 1.0)  # New parameter for fine type differentiation
        self.coarse_to_fine = kwargs.pop('coarse_to_fine', {})
        
        # Extract available_priors if present
        if 'available_priors' in kwargs:
            self.available_priors = kwargs.pop('available_priors')
        else:
            self.available_priors = None
        
        # Initialize parent class
        super().__init__(*args, **kwargs)
        
        # Initialize empty constraint dictionary
        self.constraint_dict = {}
        
        # Will store dimension mapping info
        self.fine_to_dim_mapping = {}
        
    def train(self, *args, **kwargs):
        # Set up constraints now that the model is ready
        self._setup_constraint_tensors()
        return super().train(*args, **kwargs)
        
    def _setup_constraint_tensors(self):
        """Convert constraints to torch Tensors matching device/dtype"""
        self.constraint_dict = {}
        
        # Use CPU initially
        device = torch.device('cpu')
        
        # If model is initialized, try to get its device
        if hasattr(self, 'module') and hasattr(self.module, '_model'):
            try:
                for param in self.module._model.parameters():
                    device = param.device
                    break
            except:
                pass
        
        # Store tensors for each gamma dimension separately
        for ct in self.coarse_to_fine.keys():
            gamma_key = f'{ct}_gamma'
            if gamma_key in self.adata.obsm:
                gamma = self.adata.obsm[gamma_key]
                
                # Handle different gamma formats (DataFrame vs numpy array)
                if isinstance(gamma, pd.DataFrame):
                    # For each column (dimension) in the gamma matrix
                    for col in gamma.columns:
                        tensor_key = f"{ct}_{col}"
                        self.constraint_dict[tensor_key] = torch.tensor(
                            gamma[col].values,
                            device=device,
                            dtype=torch.float32
                        )
                else:
                    # For numpy arrays - handle multi-dimensional factors
                    if gamma.ndim > 1:
                        for i in range(gamma.shape[1]):
                            tensor_key = f"{ct}_dim{i}"
                            self.constraint_dict[tensor_key] = torch.tensor(
                                gamma[:, i],
                                device=device,
                                dtype=torch.float32
                            )
                    else:
                        # Single-dimensional case
                        tensor_key = f"{ct}_dim0"
                        self.constraint_dict[tensor_key] = torch.tensor(
                            gamma.flatten(),
                            device=device,
                            dtype=torch.float32
                        )
    
    def forward(self):
        """Enhanced forward method with linear combination of dimensions and orthogonality constraints"""
        # Get parent ELBO
        elbo = super().forward()
        
        # Skip if cell abundances not initialized
        if not hasattr(self, 'sample_cell_abundance') or self.sample_cell_abundance is None:
            return elbo
        
        # Initialize constraint losses
        constraint_loss = 0.0
        orthogonality_loss = 0.0
        
        # Process each coarse cell type
        for ct, gamma_tensors in self._get_grouped_tensors().items():
            # Skip if no dimensions or no fine types
            if ct not in self.coarse_to_fine or not gamma_tensors:
                continue
                
            fine_types = self.coarse_to_fine[ct]
            idx = [self.cell_state_df.columns.get_loc(ft) for ft in fine_types 
                  if ft in self.cell_state_df.columns]
            
            if not idx:
                continue
                
            # 1. TOTAL AMOUNT CONSTRAINT - sum of fine types should match overall gamma
            subtype_sum = self.sample_cell_abundance[:, idx].sum(1)
            primary_gamma = torch.abs(gamma_tensors[0])
            amount_constraint = torch.exp(-0.5 * (subtype_sum - primary_gamma)**2 * self.constraint_scale)
            constraint_loss += torch.mean(1 - amount_constraint)
            
            # 2. LINEAR COMBINATION OF DIMENSIONS
            n_subtypes = len(idx)
            n_dims = len(gamma_tensors)
            
            # Compute correlation matrix
            correlation_matrix = torch.zeros((n_subtypes, n_dims), device=self.sample_cell_abundance.device)
            
            for i, cell_idx in enumerate(idx):
                cell_abundance = self.sample_cell_abundance[:, cell_idx]
                cell_mean = torch.mean(cell_abundance)
                cell_std = torch.std(cell_abundance) + 1e-10
                cell_std_vals = (cell_abundance - cell_mean) / cell_std
                
                for j, gamma in enumerate(gamma_tensors):
                    gamma_mean = torch.mean(gamma)
                    gamma_std = torch.std(gamma) + 1e-10
                    gamma_std_vals = (gamma - gamma_mean) / gamma_std
                    correlation_matrix[i, j] = torch.sum(cell_std_vals * gamma_std_vals) / len(gamma_std_vals)
            
            # For each fine cell type, create a linear combination of gamma dimensions
            for i, cell_idx in enumerate(idx):
                cell_abundance = self.sample_cell_abundance[:, cell_idx]
                
                # Calculate weights for linear combination based on correlation strengths
                # Use softmax to create a probability distribution over dimensions
                abs_corrs = torch.abs(correlation_matrix[i])
                weights = torch.nn.functional.softmax(abs_corrs * 5.0, dim=0)  # Temperature of 5.0 makes it more peaked
                
                # Create linear combination of gamma dimensions
                combined_gamma = torch.zeros_like(gamma_tensors[0])
                
                # Top-k approach: Only use dimensions with top-k correlations (e.g., top 2)
                k = min(2, n_dims)  # Use at most 2 dimensions
                top_k_values, top_k_indices = torch.topk(abs_corrs, k)
                
                # Get specific signs for each dimension
                signs = torch.sign(correlation_matrix[i, top_k_indices])
                
                # Create new weights that sum to 1 for just the top-k dimensions
                top_k_weights = torch.nn.functional.softmax(top_k_values * 5.0, dim=0)
                
                # Construct combined gamma using only top-k dimensions with appropriate signs
                for j, (dim_idx, weight, sign) in enumerate(zip(top_k_indices, top_k_weights, signs)):
                    combined_gamma += weight * sign * gamma_tensors[dim_idx]
                
                # Scale combined gamma to match abundance scale
                cell_mean_abs = torch.mean(torch.abs(cell_abundance)) + 1e-10
                gamma_mean_abs = torch.mean(torch.abs(combined_gamma)) + 1e-10
                scale_factor = cell_mean_abs / gamma_mean_abs
                scaled_gamma = combined_gamma * scale_factor
                
                # Apply constraint using the combined gamma
                dim_constraint = torch.exp(-0.5 * (cell_abundance - scaled_gamma)**2 * self.constraint_scale)
                constraint_loss += torch.mean(1 - dim_constraint)
                
                # Store mapping info for debugging
                if not self.training and i < len(fine_types) and fine_types[i] in self.cell_state_df.columns:
                    self.fine_to_dim_mapping[fine_types[i]] = {
                        'coarse_type': ct,
                        'top_dims': top_k_indices.cpu().numpy().tolist(),
                        'weights': top_k_weights.cpu().numpy().tolist(),
                        'signs': signs.cpu().numpy().tolist()
                    }
            
            # 3. ORTHOGONALITY CONSTRAINT - Encourage different fine types to use different patterns
            if n_subtypes > 1:
                # Calculate pairwise orthogonality penalty between fine cell types
                for i in range(n_subtypes):
                    for j in range(i+1, n_subtypes):
                        # Get the two cell abundances
                        abundance_i = self.sample_cell_abundance[:, idx[i]]
                        abundance_j = self.sample_cell_abundance[:, idx[j]]
                        
                        # Normalize to unit vectors for proper orthogonality calculation
                        norm_i = torch.sqrt(torch.sum(abundance_i**2) + 1e-10)
                        norm_j = torch.sqrt(torch.sum(abundance_j**2) + 1e-10)
                        
                        unit_i = abundance_i / norm_i
                        unit_j = abundance_j / norm_j
                        
                        # Calculate dot product (should be close to 0 if orthogonal)
                        dot_product = torch.abs(torch.sum(unit_i * unit_j))
                        
                        # Add to orthogonality loss - penalize similar patterns
                        orthogonality_loss += dot_product
            
            # Normalize orthogonality loss by number of pairs
            if n_subtypes > 1:
                n_pairs = n_subtypes * (n_subtypes - 1) // 2  # Number of unique pairs
                orthogonality_loss = orthogonality_loss / n_pairs
        
        # Add sparsity loss
        sparsity_loss = self._calculate_sparsity()
        
        # Return combined loss with all components
        return elbo - constraint_loss - sparsity_loss * self.sparsity_scale - orthogonality_loss * self.orthogonality_scale
    
    def _calculate_sparsity(self):
        """Calculate sparsity loss using L1/L2 ratio"""
        sparsity_loss = 0.0
        for i in range(self.sample_cell_abundance.shape[1]):
            subtype_vals = self.sample_cell_abundance[:, i]
            
            # Skip if all values are near zero
            if torch.mean(subtype_vals) < 1e-5:
                continue
                
            # L1/L2 ratio encourages sparsity
            l1 = torch.sum(torch.abs(subtype_vals) + 1e-10)
            l2 = torch.sqrt(torch.sum(subtype_vals**2) + 1e-10)
            ratio = l1 / l2
            sparsity_loss += ratio
            
        return sparsity_loss / self.sample_cell_abundance.shape[1]
    
    def _get_grouped_tensors(self):
        """Group tensors by cell type, handling cell types with underscores"""
        grouped = {}
        
        # Get all coarse cell types
        coarse_types = list(self.coarse_to_fine.keys())
        
        for key, tensor in self.constraint_dict.items():
            # Find the matching coarse type in the key
            matching_ct = None
            for ct in coarse_types:
                # Check if this coarse type is a prefix of the key
                if key.startswith(ct + '_'):
                    matching_ct = ct
                    break
            
            # If found a matching coarse type, add to grouped dict
            if matching_ct:
                if matching_ct not in grouped:
                    grouped[matching_ct] = []
                grouped[matching_ct].append(tensor)
        
        return grouped
    
    def get_dimension_mapping(self):
        """Return the mapping between fine cell types and gamma dimensions"""
        return pd.DataFrame.from_dict(self.fine_to_dim_mapping, orient='index')
    
    
def analyze_destvi_dimensions(sc_data, visium_data, coarse_to_fine):
    """Analyze DestVI gamma dimensions to determine their biological meaning"""
    print("\nAnalyzing DestVI cell state dimensions...")
    
    # Dictionary to store fine cell type to gamma dimension mappings
    fine_to_gamma_dim = {}
    
    # For each coarse cell type
    for coarse_type, fine_types in coarse_to_fine.items():
        gamma_key = f'{coarse_type}_gamma'
        if gamma_key not in visium_data.obsm:
            continue
            
        print(f"\nAnalyzing {coarse_type} gamma dimensions...")
        
        # Get gamma factors
        gamma_values = visium_data.obsm[gamma_key]
        
        # Get dimension names
        if isinstance(gamma_values, pd.DataFrame):
            gamma_dims = gamma_values.columns
        else:
            # If numpy array, create dimension names
            n_dims = gamma_values.shape[1] if gamma_values.ndim > 1 else 1
            gamma_dims = [f"dim{i}" for i in range(n_dims)]
        
        # Get cells belonging to this coarse type
        coarse_cells = sc_data[sc_data.obs['coarse_celltype'] == coarse_type]
        
        if len(coarse_cells) == 0:
            print(f"  No cells found for {coarse_type}")
            continue
            
        # For each fine cell type, find its marker genes
        for fine_type in fine_types:
            fine_cells = sc_data[sc_data.obs['celltypes'] == fine_type]
            
            if len(fine_cells) == 0:
                print(f"  No cells found for {fine_type}")
                continue
                
            # Get marker genes for this fine cell type vs others in same coarse type
            other_fine_cells = coarse_cells[coarse_cells.obs['celltypes'] != fine_type]
            
            if len(other_fine_cells) == 0:
                # If this is the only fine cell type in the coarse type
                # Skip creating marker genes - can't compare against itself
                print(f"  {fine_type} is the only subtype in {coarse_type}, using default mapping")
                best_dim = gamma_dims[0] if len(gamma_dims) > 0 else None
                best_score = 0.5  # Neutral score
            else:
                # Convert boolean to categorical for rank_genes_groups
                temp = coarse_cells.copy()
                temp.obs['is_target'] = (temp.obs['celltypes'] == fine_type).astype(str).astype('category')
                
                try:
                    # ALWAYS create a log-transformed copy for marker gene analysis
                    temp_log = temp.copy()
                    
                    # Save raw counts if needed later
                    if 'counts' not in temp_log.layers and not scipy.sparse.issparse(temp_log.X):
                        temp_log.layers['counts'] = temp_log.X.copy()
                    
                    # Always log-transform - scanpy will skip if already log-transformed
                    print(f"  Log-transforming data for {fine_type}")
                    
                    # First normalize to account for library size differences
                    sc.pp.normalize_total(temp_log, target_sum=1e4)
                    
                    # Then log-transform
                    sc.pp.log1p(temp_log)
                    
                    # Run rank_genes_groups on log-transformed data
                    marker_genes = sc.tl.rank_genes_groups(temp_log, 
                                                         groupby='is_target', 
                                                         groups=['True'], 
                                                         reference='rest',
                                                         n_genes=50,
                                                         method='wilcoxon',
                                                         copy=True)
                
                    # Extract top marker genes
                    if hasattr(marker_genes, 'uns') and 'rank_genes_groups' in marker_genes.uns:
                        marker_names = marker_genes.uns['rank_genes_groups']['names']['True'][:20]
                        
                        # Check which gamma dimension best correlates with these marker genes
                        best_dim = None
                        best_score = -float('inf')
                        
                        # For each gamma dimension, calculate correlation with marker genes
                        for dim_idx, dim_name in enumerate(gamma_dims):
                            # Extract gamma values for this dimension
                            if isinstance(gamma_values, pd.DataFrame):
                                dim_values = gamma_values[dim_name].values
                            else:
                                dim_values = gamma_values[:, dim_idx] if gamma_values.ndim > 1 else gamma_values
                            
                            # Calculate correlation with marker gene expression
                            dim_score = calculate_marker_correlation(
                                visium_data, dim_values, marker_names)
                            
                            print(f"  {fine_type} correlation with {dim_name}: {dim_score:.3f}")
                            
                            if dim_score > best_score:
                                best_score = dim_score
                                best_dim = dim_name
                    else:
                        print(f"  No marker genes found for {fine_type}")
                        best_dim = gamma_dims[0] if len(gamma_dims) > 0 else None
                        best_score = 0.0
                
                except Exception as e:
                    print(f"  Error finding markers for {fine_type}: {str(e)}")
                    best_dim = gamma_dims[0] if len(gamma_dims) > 0 else None
                    best_score = 0.0
            
            # Store the mapping
            if best_dim is not None:
                fine_to_gamma_dim[fine_type] = {
                    'coarse_type': coarse_type,
                    'gamma_dim': best_dim,
                    'score': best_score
                }
                print(f"  Mapped {fine_type} to {coarse_type}_{best_dim} (score: {best_score:.3f})")
    
    return fine_to_gamma_dim

def calculate_marker_correlation(adata, gamma_values, marker_genes):
    """Calculate correlation between gamma values and marker gene expression"""
    # Get expression of marker genes
    common_markers = [gene for gene in marker_genes if gene in adata.var_names]
    if not common_markers:
        return 0.0
        
    # Calculate average expression of markers in each spot
    marker_expr = adata[:, common_markers].X
    if scipy.sparse.issparse(marker_expr):
        marker_expr = marker_expr.toarray()
        
    # Calculate average expression across markers
    marker_avg = np.mean(marker_expr, axis=1)
    
    # Calculate correlation
    try:
        corr, _ = pearsonr(gamma_values, marker_avg)
        return corr
    except:
        return 0.0

def assess_expression_reconstruction(visium_data, model, reference_signatures):
    """Assess how well the model reconstructs the observed gene expression"""
    # Get predicted counts
    predicted_counts = model.get_likelihood_parameters()[0].detach().cpu().numpy()
    
    # Get observed counts
    observed_counts = visium_data.X.toarray() if scipy.sparse.issparse(visium_data.X) else visium_data.X
    
    # Calculate correlation for each spot
    spot_corrs = []
    for i in range(observed_counts.shape[0]):
        corr = np.corrcoef(predicted_counts[i], observed_counts[i])[0, 1]
        if not np.isnan(corr):
            spot_corrs.append(corr)
    
    # Return average correlation
    return np.mean(spot_corrs)

def plot_cell_types(adata, cell_type, ax=None, spot_size=None):
    """Plot abundance of a cell type on the spatial coordinates"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10))
    
    if cell_type in adata.obsm['means_cell_abundance_w_sf'].columns:
        # Get abundances
        abundances = adata.obsm['means_cell_abundance_w_sf'][cell_type].values
        
        # Calculate spot size if not provided
        if spot_size is None:
            spot_size = 100000 / adata.n_obs
        
        # Plot abundances
        sc.pl.spatial(
            adata, 
            color=[],
            show=False,
            ax=ax, 
            spot_size=spot_size,
        )
        
        s = ax.scatter(
            adata.obsm['spatial'][:, 0],
            adata.obsm['spatial'][:, 1],
            c=abundances, 
            s=spot_size, 
            cmap='viridis',
            vmin=0,
            vmax=np.percentile(abundances, 99.5)
        )
        
        plt.colorbar(s, ax=ax, shrink=0.5, label=f'{cell_type} abundance')
        ax.set_title(f'{cell_type}')
        ax.set_axis_off()
        
    return ax

def plot_fine_clusters(adata, condition, slice_to_plot=None, output_dir="./results_adaptive"):
    """Plot fine cluster deconvolution results using cell2location plotting"""
    import matplotlib.pyplot as plt
    import os
    from pathlib import Path
    
    # Create output directory if it doesn't exist
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # Filter by slice if specified
    if slice_to_plot is not None:
        slice_data = adata[adata.obs.index.str.contains(slice_to_plot)].copy()
        if slice_data.shape[0] == 0:
            print(f"No spots found for slice {slice_to_plot}")
            return
    else:
        slice_data = adata.copy()
    
    # Get cell abundance data
    if 'means_cell_abundance_w_sf' not in slice_data.obsm:
        print("Cell abundance not found in data")
        return
    
    # Get top cell types by total abundance
    cell_abundance = slice_data.obsm['means_cell_abundance_w_sf']
    cell_sums = cell_abundance.sum(axis=0).sort_values(ascending=False)
    cell_types = cell_sums.index[:45].tolist()  # Top 20 cell types
    
    print(f"Plotting {len(cell_types)} cell types for {condition} {slice_to_plot}")
    
    # Plot each cell type individually
    for cell_type in cell_types:
        try:
            # Temporarily add this cell type to obs
            slice_data.obs[cell_type] = slice_data.obsm['means_cell_abundance_w_sf'][cell_type]
           
            with plt.rc_context({'figure.figsize': (15, 15)}):
                fig = cell2location.plt.plot_spatial(
                    adata=slice_data,
                    color=[cell_type],
                    labels=[cell_type],
                    show_img=True,
                    style='fast',
                    max_color_quantile=0.992,
                    circle_diameter=6,
                    colorbar_position='right'
                )
               
                # Save figure
                safe_cell_type = cell_type.replace('/', '-')
                output_file = output_dir / f"{condition}_{slice_to_plot}_{safe_cell_type}.png"
                plt.savefig(
                    output_file,
                    dpi=300,
                    bbox_inches='tight',
                    facecolor='white'
                )
                plt.close()
           
            # Remove the temporary column
            del slice_data.obs[cell_type]
               
        except Exception as e:
            print(f"Error plotting {cell_type}: {str(e)}")
            plt.close()
            continue
   
    print(f"Individual fine cluster plots saved to {output_dir}")
    
def calculate_constraint_impact(model):
    """Compare constrained vs unconstrained posteriors with proper retraining"""
    import copy
    import gc
    
    print("\nCalculating constraint impact (this may take a few minutes)...")
    
    # Get constrained abundances
    constrained_abundance = model.adata.obsm['means_cell_abundance_w_sf']
    
    try:
        # Create a deep copy of the model with disabled constraints
        temp_model = copy.deepcopy(model)
        temp_model.constraint_scale = 0.0
        
        # Keep same data reference to avoid duplication
        adata_copy = model.adata.copy()
        
        # Run short training with constraints disabled
        from lightning.pytorch.callbacks import EarlyStopping
        temp_model.train(
            max_epochs=30000,  # Short training to adapt to unconstrained mode
            early_stopping=EarlyStopping(
                monitor="elbo_train",
                min_delta=0.001,
                patience=10,
                mode="min"
            ),
            plan_kwargs={
                'optim': pyro.optim.Adamax({'lr': 0.01, 'weight_decay': 0.01}),
            }
        )
        
        # Export unconstrained posterior
        unconstrained_adata = temp_model.export_posterior(adata_copy)
        unconstrained_abundance = unconstrained_adata.obsm['means_cell_abundance_w_sf']
        
        # Calculate divergence metrics
        divergence = {}
        for ct in constrained_abundance.columns:
            r, p = pearsonr(constrained_abundance[ct], unconstrained_abundance[ct])
            # Calculate mean absolute difference
            mad = np.mean(np.abs(constrained_abundance[ct] - unconstrained_abundance[ct]))
            # Calculate KL divergence approximation
            kl_div = np.mean(np.where(
                constrained_abundance[ct] > 0,
                constrained_abundance[ct] * np.log(
                    (constrained_abundance[ct] + 1e-6) / (unconstrained_abundance[ct] + 1e-6)
                ),
                0
            ))
            
            divergence[ct] = {
                'pearson_r': r, 
                'p_value': p,
                'mean_abs_diff': mad,
                'kl_divergence': kl_div
            }
        
        # Sort by increasing correlation (most affected first)
        print("\nConstraint Impact Analysis:")
        print("Cell Type | Correlation | Mean Abs Diff | KL Divergence")
        print("---------|-----------|--------------|--------------")
        for ct, vals in sorted(divergence.items(), key=lambda x: x[1]['pearson_r']):
            print(f"{ct}: r = {vals['pearson_r']:.2f}, diff = {vals['mean_abs_diff']:.3f}, kl = {vals['kl_divergence']:.3f}")
        
        # Create summary of most affected types
        most_affected = sorted(divergence.items(), key=lambda x: x[1]['kl_divergence'], reverse=True)[:5]
        print("\nMost affected cell types (by KL divergence):")
        for ct, vals in most_affected:
            print(f"{ct}: KL = {vals['kl_divergence']:.3f}, r = {vals['pearson_r']:.2f}")
            
        return divergence
        
    except Exception as e:
        print(f"Error in constraint impact analysis: {str(e)}")
        return {}
    
    finally:
        # Clean up memory
        if 'temp_model' in locals():
            del temp_model
        if 'unconstrained_adata' in locals():
            del unconstrained_adata
        gc.collect()
        
def validate_deconvolution(adata, coarse_to_fine, condition_name, output_dir=None):
    """Validates deconvolution results to identify potential biases"""
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    import pandas as pd
    from scipy.stats import pearsonr
    
    print(f"\nValidating deconvolution results for {condition_name}...")
    metrics = {} 
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
    
    # 1. Hierarchical consistency check
    print("Checking hierarchical consistency...")
    consistency_data = []
    
    for coarse_type, fine_types in coarse_to_fine.items():
        if f'{coarse_type}_gamma' in adata.obsm:
            # Fix: Ensure proper shape of coarse_prior
            coarse_prior = adata.obsm[f'{coarse_type}_gamma']
            
            # Handle different possible shapes of the gamma matrix
            if isinstance(coarse_prior, pd.DataFrame):
                if coarse_type in coarse_prior.columns:
                    coarse_prior = coarse_prior[coarse_type].values
                else:
                    # Just take the first column if the specific type isn't found
                    coarse_prior = coarse_prior.iloc[:, 0].values
            else:
                # If it's a numpy array, flatten it correctly
                coarse_prior = coarse_prior.flatten()
                
            # Get sum of fine cell types
            valid_fine_types = [ft for ft in fine_types if ft in adata.obsm['means_cell_abundance_w_sf'].columns]
            if not valid_fine_types:
                continue
                
            fine_sum = adata.obsm['means_cell_abundance_w_sf'][valid_fine_types].sum(axis=1).values
            
            # Double-check the shapes match before calculating correlation
            if len(coarse_prior) != len(fine_sum):
                print(f"  Warning: Shape mismatch for {coarse_type}: {coarse_prior.shape} vs {fine_sum.shape}")
                # Try to standardize sizes by taking the minimum shared length
                min_len = min(len(coarse_prior), len(fine_sum))
                coarse_prior = coarse_prior[:min_len]
                fine_sum = fine_sum[:min_len]
                
            corr, p_value = pearsonr(coarse_prior, fine_sum)
            mae = np.mean(np.abs(coarse_prior - fine_sum))
            ratio = np.mean(fine_sum) / np.mean(coarse_prior) if np.mean(coarse_prior) > 0 else np.nan
            
            # Add permutation test for statistical assessment
            n_permutations = 100
            null_corrs = [
                pearsonr(fine_sum, np.random.permutation(coarse_prior))[0] 
                for _ in range(n_permutations)
            ]
            null_mean = np.mean(null_corrs)
            null_std = np.std(null_corrs)
            
            # Calculate empirical p-value and z-score
            if corr >= null_mean:
                empirical_p = sum(null_c >= corr for null_c in null_corrs) / n_permutations
            else:
                empirical_p = sum(null_c <= corr for null_c in null_corrs) / n_permutations
                
            z_score = (corr - null_mean) / (null_std + 1e-10)  # add small constant to prevent division by zero
            
            print(f"  {coarse_type}: r={corr:.3f}, p={empirical_p:.3f}, z={z_score:.2f} (null: μ={null_mean:.3f}, σ={null_std:.3f})")
            
            consistency_data.append({
                'coarse_type': coarse_type,
                'correlation': corr,
                'p_value': p_value,
                'empirical_p': empirical_p,
                'z_score': z_score,
                'null_mean': null_mean,
                'null_std': null_std,
                'mae': mae,
                'ratio': ratio
            })
            
            # Plot comparison
            if output_dir:
                plt.figure(figsize=(10, 6))
                plt.scatter(coarse_prior, fine_sum, alpha=0.3)
                plt.title(f"{coarse_type}: r={corr:.3f}, ratio={ratio:.2f}")
                plt.xlabel("Coarse Abundance Prior")
                plt.ylabel("Sum of Fine Type Abundances")
                plt.savefig(os.path.join(output_dir, f"{coarse_type}_consistency.png"))
                plt.close()
    
    # Save metrics
    metrics['hierarchical_consistency'] = pd.DataFrame(consistency_data)
    if output_dir:
        metrics['hierarchical_consistency'].to_csv(
            os.path.join(output_dir, "hierarchical_consistency.csv"), 
            index=False
        )
    
    # 2. Calculate spatial autocorrelation (modified to avoid moran import)
    print("Calculating spatial patterns...")
    spatial_metrics = []
    
    # Get cell types from abundance matrix
    cell_types = adata.obsm['means_cell_abundance_w_sf'].columns
    
    for cell_type in cell_types:
        try:
            # Get abundance values
            abundance = adata.obsm['means_cell_abundance_w_sf'][cell_type].values
            
            # Skip if all values are the same (no variance)
            if np.all(abundance == abundance[0]):
                continue
                
            # Skip calculation if spatial_connectivities not available
            if 'spatial_connectivities' not in adata.obsp:
                print(f"  Warning: No spatial_connectivities found, skipping spatial analysis")
                break
            
            # Get spatial weights
            weights = adata.obsp['spatial_connectivities']
            weights_array = weights.toarray() if hasattr(weights, 'toarray') else weights
            
            # Calculate simple spatial correlation (neighbor similarity)
            # This is a simplified version of spatial autocorrelation
            total_similarity = 0
            total_connections = 0
            
            for i in range(len(abundance)):
                for j in range(len(abundance)):
                    if weights_array[i, j] > 0:
                        # Calculate similarity between neighbors
                        similarity = 1 - abs(abundance[i] - abundance[j]) / (max(abundance[i], abundance[j]) + 1e-10)
                        total_similarity += similarity * weights_array[i, j]
                        total_connections += weights_array[i, j]
            
            # Average similarity weighted by connection strength
            spatial_coherence = total_similarity / (total_connections + 1e-10)
            
            # Try to use pysal if available (more accurate)
            try:
                import esda
                from libpysal.weights import W
                
                # Convert scipy sparse matrix to libpysal weights
                neighbors = {}
                for i in range(weights_array.shape[0]):
                    neighbors[i] = list(np.where(weights_array[i] > 0)[0])
                
                weights_dict = {}
                for i in range(weights_array.shape[0]):
                    weights_dict[i] = weights_array[i, neighbors[i]].tolist()
                
                w = W(neighbors, weights_dict)
                
                # Calculate Moran's I
                moran = esda.moran.Moran(abundance, w)
                moran_i = moran.I
                moran_p = moran.p_sim
                print(f"  {cell_type}: Moran's I = {moran_i:.3f} (p = {moran_p:.3f})")
                
                spatial_metrics.append({
                    'cell_type': cell_type,
                    'morans_i': moran_i,
                    'p_value': moran_p,
                    'spatial_coherence': spatial_coherence
                })
                
            except ImportError:
                # Fallback to just using our simple measure
                print(f"  {cell_type}: Spatial coherence = {spatial_coherence:.3f}")
                
                spatial_metrics.append({
                    'cell_type': cell_type,
                    'spatial_coherence': spatial_coherence,
                })
                
        except Exception as e:
            print(f"  Error calculating spatial metrics for {cell_type}: {str(e)}")
    
    # Save spatial metrics
    if spatial_metrics:
        metrics['spatial_metrics'] = pd.DataFrame(spatial_metrics)
        if output_dir:
            metrics['spatial_metrics'].to_csv(
                os.path.join(output_dir, "spatial_metrics.csv"), 
                index=False
            )
    
    # 3. Create a summary report
    print("\nDeconvolution Validation Summary:")
    
    # Print hierarchical consistency
    if 'hierarchical_consistency' in metrics:
        df = metrics['hierarchical_consistency']
        print("\nHierarchical Consistency:")
        for _, row in df.iterrows():
            print(f"  {row['coarse_type']}: r={row['correlation']:.3f}, MAE={row['mae']:.3f}, ratio={row['ratio']:.2f}")
    
    # Print spatial coherence 
    if 'spatial_coherence' in metrics:
        df = metrics['spatial_coherence'].sort_values('morans_i', ascending=False)
        print("\nSpatial Coherence (top 10, higher = more spatially clustered):")
        for _, row in df.head(10).iterrows():
            print(f"  {row['cell_type']}: {row['morans_i']:.3f}")
    
    return metrics

def run_adaptive_deconvolution(visium_data, sc_data, condition_name, coarse_to_fine, output_dir="./results"):
    """Run cell2location with hierarchical cell type constraints
    
    Args:
        visium_data: AnnData object with Visium data
        sc_data: AnnData object with single-cell reference data
        condition_name: Name of condition for logging/output
        coarse_to_fine: Dictionary mapping coarse cell types to lists of fine cell types
        output_dir: Path to save outputs (default: "./results")
    """
    print(f"\nRunning adaptive deconvolution for {condition_name} condition...")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    # Analyze DestVI dimensions to map fine cell types to appropriate dimensions
    fine_to_gamma_dim = analyze_destvi_dimensions(sc_data, visium_data, coarse_to_fine)
    
    # Identify available destVI priors 
    available_priors = []
    for key in visium_data.obsm.keys():
        if key.endswith('_gamma'):
            coarse_type = key.replace('_gamma', '')
            if coarse_type in coarse_to_fine:
                available_priors.append(coarse_type)
                print(f"  Using {key} as spatial prior")
    
    # Train reference model
    print("\nTraining reference model...")
    cell2location.models.RegressionModel.setup_anndata(
        sc_data,
        batch_key='batch',
        labels_key='celltypes'
    )
    
    ref_model = cell2location.models.RegressionModel(sc_data)
    ref_model.train(
        max_epochs=30000,
        accelerator="gpu" if torch.cuda.is_available() else "cpu"
    )
    
    # Export reference model results
    sc_data = ref_model.export_posterior(sc_data)
    # Prepare data for spatial model
    print("\nPreparing for spatial deconvolution...")
    
    # Match genes between reference and spatial data
    common_genes = visium_data.var_names.intersection(sc_data.var_names)
    visium_data = visium_data[:, common_genes].copy()
    sc_data = sc_data[:, common_genes].copy()
    
    # Setup for cell2location
    cell2location.models.Cell2location.setup_anndata(visium_data)
    
    # Set up anndata for the model
    print("\nSetting up AnnData for spatial model...")
    AdaptiveConstraintModel.setup_anndata(visium_data)
    # ===============================================
    
    # Train spatial model with adaptive constraints
    print("\nTraining spatial model with adaptive constraints...")
    
    model = AdaptiveConstraintModel(
        visium_data,
        cell_state_df=sc_data.varm['means_per_cluster_mu_fg'],
        N_cells_per_location=30,  # Standard for 50μm spots
        detection_alpha=50,
        available_priors=available_priors,
        coarse_to_fine=coarse_to_fine,
    )
    
    from lightning.pytorch.callbacks import EarlyStopping  # Updated import path
    # Define early stopping callback
    early_stop_callback = EarlyStopping(monitor="elbo_train", patience=50, mode="max")

    # Use in training
    model.train(
        max_epochs=30000,
        batch_size=1024,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        plan_kwargs={
            'optim': pyro.optim.Adamax({'lr': 0.01, 'weight_decay': 0.01}),
            'scale_elbo': 1.0  # This is a standard parameter mentioned in the docs
        },
        callbacks=[early_stop_callback]  # Pass the callback directly
    )
    
    # Export posterior
    # Export posterior
    print("\nExporting posterior distributions...")
    visium_data = model.export_posterior(
        visium_data
    )
      
    # Run constraint impact analysis
    constraint_impact = calculate_constraint_impact(model)
    
    # Save impact analysis to file
    impact_df = pd.DataFrame.from_dict(constraint_impact, orient='index')
    impact_file = os.path.join(output_dir, f"{condition_name}_constraint_impact.csv")
    impact_df.to_csv(impact_file)
    
    # Clean column names
    for key_prefix in ['means', 'q05']:
        key = f'{key_prefix}_cell_abundance_w_sf'
        if key in visium_data.obsm:
            visium_data.obsm[key].columns = (
                visium_data.obsm[key].columns
                .str.replace(f'{key_prefix}cell_abundance_w_sf_means_per_cluster_mu_fg_', '')
            )
    
    
    # Compare with destVI for validation
    print("\nComparing with destVI for validation...")
    
    for coarse_type in available_priors:
        fine_types = coarse_to_fine.get(coarse_type, [])
        if not fine_types:
            continue
            
        available_fine = [ft for ft in fine_types 
                         if ft in visium_data.obsm['means_cell_abundance_w_sf'].columns]
        if not available_fine:
            continue
            
        # Calculate total abundance for this coarse type
        total_abundance = visium_data.obsm['means_cell_abundance_w_sf'][available_fine].sum(axis=1)
        
        # Get destVI gamma values 
        gamma_values = visium_data.obsm[f'{coarse_type}_gamma']

        # Check if gamma_values is a DataFrame with multiple columns
        if isinstance(gamma_values, pd.DataFrame) and gamma_values.shape[1] > 1:
            # Take the sum across columns if there are multiple dimensions
            gamma_values = gamma_values.sum(axis=1).values
        else:
            # Otherwise just get the values
            gamma_values = gamma_values.values

        # Ensure both arrays have same length
        if len(gamma_values) != len(total_abundance):
            print(f"Warning: Length mismatch for {coarse_type}. Skipping correlation calculation.")
            corr = float('nan')
        else:
            # Calculate correlation
            corr = pearsonr(total_abundance, gamma_values)[0]
        print(f"  {coarse_type}: Correlation with destVI = {corr:.3f}")
        
        # Store top subtypes by abundance
        subtype_means = {
            subtype: visium_data.obsm['means_cell_abundance_w_sf'][subtype].mean()
            for subtype in available_fine
        }
        top_subtypes = sorted(subtype_means.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"    Top subtypes: {', '.join(f'{st} ({val:.2f})' for st, val in top_subtypes)}")
    
    # Add metadata
    visium_data.obs['sample'] = visium_data.obs.index.str.split('_').str[4]
    visium_data.obs['condition'] = condition_name
    
    validation_dir = os.path.join(output_dir, f"{condition_name}_validation")
    validation_metrics = validate_deconvolution(
        visium_data, 
        coarse_to_fine,
        condition_name,
        output_dir=validation_dir
    )
    
    # Save validation metrics in a readable format
    metrics_df = pd.DataFrame()
    
    # Combine metrics for all cell types
    for cell_type in visium_data.obsm['means_cell_abundance_w_sf'].columns:
        record = {'cell_type': cell_type}
        
        # Add spatial coherence if available
        if 'spatial_coherence' in validation_metrics:
            sc_data = validation_metrics['spatial_coherence']
            match = sc_data[sc_data['cell_type'] == cell_type]
            if not match.empty:
                record['spatial_coherence'] = float(match['morans_i'].values[0])
        
        # Add hierarchical relationship if available
        for coarse_type, fine_types in coarse_to_fine.items():
            if cell_type in fine_types:
                record['parent_type'] = coarse_type
                break
        
        metrics_df = pd.concat([metrics_df, pd.DataFrame([record])], ignore_index=True)
    
    # Save readable metrics
    metrics_df.to_csv(os.path.join(validation_dir, "cell_type_metrics.csv"), index=False)
    
    return visium_data
    

def resolve_gene_expression(visium_data, sc_data):
    """Resolve gene expression for each cell type"""
    print("\nResolving gene expression per cell type...")
    
    if 'means_cell_abundance_w_sf' not in visium_data.obsm:
        print("Cell abundance not found in visium data")
        return visium_data
    
    # Get cell abundance matrix
    cell_abundance = visium_data.obsm['means_cell_abundance_w_sf']
    
    # Get available cell types
    cell_types = cell_abundance.columns
    
    # Extract gene expression for each cell type from reference
    cell_type_expr = {}
    for cell_type in cell_types:
        if cell_type in sc_data.obs['celltypes'].unique():
            # Get cells of this type
            cells = sc_data[sc_data.obs['celltypes'] == cell_type]
            
            # Skip if no cells found
            if cells.n_obs == 0:
                continue
                
            # Calculate average expression
            if scipy.sparse.issparse(cells.X):
                expr = cells.X.mean(axis=0).A1
            else:
                expr = cells.X.mean(axis=0)
                
            cell_type_expr[cell_type] = expr
    
    # Create per-celltype expression layer
    for cell_type, expr in cell_type_expr.items():
        layer_name = f"{cell_type}_expression"
        visium_data.layers[layer_name] = np.outer(
            cell_abundance[cell_type].values, expr
        )
    
    # Create total expression layer
    visium_data.layers['celltype_expression'] = np.zeros_like(visium_data.X, dtype=float)
    for cell_type in cell_type_expr:
        layer_name = f"{cell_type}_expression"
        if layer_name in visium_data.layers:
            visium_data.layers['celltype_expression'] += visium_data.layers[layer_name]
    
    return visium_data

def main():
    # Set paths to data files
    visium_path = "/Users/yashkulkarni/cellxgene_data/1_18_25_visium_annotated.h5ad"
    sc_path = "/Users/yashkulkarni/cellxgene_data/processed_deconvolution_sc_spleen_240713_fixed_full_data.h5ad"
    output_dir = "./deconvolution_results_adaptive"  # Output directory
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Load and split data
        print("Loading and splitting data...")
        visium_uninfected, visium_infected, sc_uninfected, sc_infected, hierarchy_map, coarse_to_fine = load_and_split_data(visium_path, sc_path)
        
        visium_uninfected = filter_problematic_genes(visium_uninfected)
        visium_infected = filter_problematic_genes(visium_infected)
        sc_uninfected = filter_problematic_genes(sc_uninfected)
        sc_infected = filter_problematic_genes(sc_infected)
        

        print("\nRunning deconvolution for uninfected samples...")
        visium_uninfected = run_adaptive_deconvolution(
            visium_uninfected, sc_uninfected, "uninfected", coarse_to_fine, output_dir
        )
        
        print("\nRunning deconvolution for infected samples...")
        visium_infected = run_adaptive_deconvolution(
            visium_infected, sc_infected, "infected", coarse_to_fine, output_dir
        )
        
        
        # Plot fine clusters for each condition and slice
        plot_dir = os.path.join(output_dir, "plots")
        print("\nGenerating plots...")
        
        # For infected condition
        for slice_id in ['V1S1', 'V1S2']:
            plot_fine_clusters(visium_infected, "infected", slice_to_plot=slice_id, output_dir=plot_dir)
        
        # For uninfected condition
        for slice_id in ['V1S3', 'V1S4']:
            plot_fine_clusters(visium_uninfected, "uninfected", slice_to_plot=slice_id, output_dir=plot_dir)
        
        # Save results
        print("\nSaving results...")
        results_dir = os.path.join(output_dir, "h5ad")
        os.makedirs(results_dir, exist_ok=True)
        
        visium_uninfected.write(f"{results_dir}/uninfected_adaptive_deconvolution.h5ad")
        visium_infected.write(f"{results_dir}/infected_adaptive_deconvolution.h5ad")
        print(f"\nAnalysis complete! Results saved to {output_dir}")
        
    except Exception as e:
        print(f"\nError during execution: {str(e)}")
        raise
    finally:
        # Clean up memory
        print("\nCleaning up memory...")
        import gc
        gc.collect()

if __name__ == "__main__":
    main()

Loading and splitting data...
Loading and preprocessing data...
Creating hierarchical cell type annotations...

Hierarchical structure:
Erythrocyte: 2 subtypes
Myeloid: 10 subtypes
CD8-Tcell: 9 subtypes
CD4-Tcell: 6 subtypes
NK: 3 subtypes
Neutrophil: 2 subtypes
Thrombocyte: 1 subtypes
Bcell: 6 subtypes
Mast-Cell: 1 subtypes
Hematopoietic: 1 subtypes
Endothelial: 1 subtypes
Fibroblast: 3 subtypes

Running deconvolution for uninfected samples...

Running adaptive deconvolution for uninfected condition...

Analyzing DestVI cell state dimensions...

Analyzing Myeloid gamma dimensions...
  Log-transforming data for Myeloid_marginalzone
  Myeloid_marginalzone correlation with 0: 0.651
  Myeloid_marginalzone correlation with 1: -0.593
  Myeloid_marginalzone correlation with 2: -0.007
  Myeloid_marginalzone correlation with 3: 0.332
  Myeloid_marginalzone correlation with 4: 0.688
  Mapped Myeloid_marginalzone to Myeloid_4 (score: 0.688)
  Log-transforming data for Myeloid_DC2
  Myeloid_DC2 c

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/homebrew/anaconda3/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/opt/homebrew/anaconda3/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
/opt/homebrew/anaconda3/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training:   0%|          | 0/10 [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...



Error during execution: name 'exit' is not defined

Cleaning up memory...


NameError: name 'exit' is not defined